# Sinh audio m1-v1 trên Kaggle

Notebook tạo WAV từ lời khách hàng, dùng bảng giới tính và tỉnh của bộ cũ. SC-071–SC-100
được trộn nền thật DEMAND sau khi sinh WAV sạch. Mỗi lượt khách hàng có một file WAV.

Input gồm 100 kịch bản, 211 cuộc gọi và 805 lượt, trong đó 255 lượt thêm nhiễu.
Lời lấy từ m1-v1, theo tên tệp SC-001.json đến SC-100.json. Bảng kết quả ghi cả mã gốc trong JSON và mã tệp. Reference ViMD dùng đúng bảng cũ theo mã tệp. Cột xưng hô cho biết khi giới tính giọng khác với transcript. Gói input có sẵn reference và nhiễu DEMAND.

Upload gói `m1_v1_audio_kaggle_input.zip` thành Dataset riêng tư rồi chọn Add Input.
Bật Internet và GPU trong Session options. Chạy tất cả các ô. Output gồm ZIP audio và bảng phân bổ.


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

# Kiểm tra CUDA trong subprocess để pip không đổi thư viện đang được import.
subprocess.check_call([sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(), "Bật GPU trong Session options"; '
    'print(torch.__version__, torch.cuda.get_device_name(0))'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'omnivoice==0.2.1', 'transformers==5.3.0', 'soundfile',
    'pandas', 'pyarrow', 'requests', 'scipy',
])


In [ ]:
from pathlib import PurePosixPath
from hashlib import sha256

INPUTS = Path('/kaggle/input')
WORK = Path('/kaggle/working') / 'm1_v1_SC001_SC100_01f2dca7'
SEED_DIR = Path('/tmp') / WORK.name
INPUT_ZIP = 'm1_v1_audio_kaggle_input.zip'
PLAN_SHA256 = 'fce104605e5e65b258742a1217da562bc2e2e27b1e2d26604004295c0db3bc95'
WORK.mkdir(parents=True, exist_ok=True)
files = [path for path in INPUTS.rglob('audio_plan.json')
         if sha256(path.read_bytes()).hexdigest() == PLAN_SHA256]
if len(files) == 1:
    bundle = files[0].parent
elif not files:
    archives = list(INPUTS.rglob(INPUT_ZIP))
    if len(archives) != 1:
        raise RuntimeError(f'Add Input một gói {INPUT_ZIP}.')
    with zipfile.ZipFile(archives[0]) as archive:
        if sha256(archive.read('audio_plan.json')).hexdigest() != PLAN_SHA256:
            raise ValueError('Gói input không khớp với notebook này.')
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if info.is_dir():
                continue
            if relative.is_absolute() or '..' in relative.parts or chr(92) in info.filename or ':' in info.filename:
                raise ValueError(f'Đường dẫn trong ZIP không hợp lệ: {info.filename}')
            destination = SEED_DIR / Path(*relative.parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as original, destination.open('wb') as target:
                shutil.copyfileobj(original, target)
    bundle = SEED_DIR
else:
    raise RuntimeError('Có nhiều gói sinh audio. Chỉ giữ một nguồn input.')

for name in ['references', 'scenarios_complete', 'cache', 'runtime']:
    source = bundle / name
    destination = WORK / name
    if source.is_dir():
        for path in source.rglob('*'):
            if path.is_file():
                target = destination / path.relative_to(source)
                if not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(path, target)
shutil.copy2(bundle / 'audio_plan.json', WORK / 'audio_plan.json')
sys.path.insert(0, str(bundle / 'scripts'))
os.environ['HF_HOME'] = '/tmp/regen_audio_hf'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '90'
import regenerate_transcript_audio as pipeline

plan = pipeline.read_plan(WORK)
print('Số WAV dự kiến:', len(plan['tasks']))


In [ ]:
from huggingface_hub import snapshot_download
import json

if plan.get('source_dataset'):
    bank = json.loads((WORK / 'references/reference_bank.json').read_text(encoding='utf-8'))
    key = pipeline.old_helpers()['province_key']
    bad = [sid for sid, voice in plan['voices'].items()
           if not pipeline.reference_valid(WORK, bank.get(sid), voice, key)]
    if bad:
        raise ValueError(f'Reference tải lên bị thiếu, hỏng hoặc sai nhãn: {bad}. Attach lại gói đúng.')
    print(f'Đã xác minh {len(bank)} reference cũ.')
else:
    pipeline.prepare_references(WORK, WORK, workers=6)
MODEL_DIR = Path('/tmp/regen_audio_model')
snapshot_download(
    'splendor1811/omnivoice-vietnamese',
    revision='f70093f90d78a5b790ff0f066b3629ebcc33cb5d',
    local_dir=MODEL_DIR, allow_patterns=['*.json','*.safetensors','*.jinja'],
)
snapshot_download(
    'k2-fsa/OmniVoice',
    revision='c5fdb5ccb189668d56333f77ba2629f4cd7535f4',
    local_dir=MODEL_DIR, allow_patterns=['audio_tokenizer/**'],
)


In [ ]:
NUM_STEPS = 16
MAX_TURNS = None  # Đặt 3 để thử. Đặt lại None rồi chạy lại ô này để sinh đủ.
pipeline.generate(WORK, MODEL_DIR, limit=MAX_TURNS, num_steps=NUM_STEPS)


In [ ]:
if MAX_TURNS is None:
    pipeline.finalize(WORK, WORK / 'runtime/demand16k')
    print(WORK / 'audio_SC001_SC100.zip')
    print(WORK / 'result/audio_allocation.csv')
    print(WORK / 'result/audio_allocation.md')
else:
    from IPython.display import Audio, display
    for row in plan['tasks'][:MAX_TURNS]:
        wav = WORK / 'clean' / row['path']
        if wav.is_file():
            print(row['task_id'])
            display(Audio(filename=str(wav)))


Trong Kaggle dùng **Save Version → Save & Run All** để lưu kết quả cả phiên.
File `m1_v1_SC001_SC100_01f2dca7/audio_SC001_SC100.zip` chỉ xuất hiện khi đã xác minh đủ WAV.
Bảng audio_allocation.csv ghi nguồn lời, giọng mẫu ViMD và SNR từng lượt có nền.
